[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/weaviate/recipes/blob/main/weaviate-features/model-providers/minimax/rag_minimax_m2.7.ipynb)

# Generative Search with MiniMax

In this demo, we will use Weaviate's Embedding Service to index blog posts and use MiniMax's `MiniMax-M2.7` model to answer questions over the retrieved content.

**What you will build:**
1. A Weaviate collection to store and search blog chunks using Weaviate's built-in embedding service.
2. A RAG pipeline that retrieves the top-K chunks from Weaviate and sends them to MiniMax for generation.

MiniMax provides an [OpenAI-compatible Chat API](https://platform.minimax.io/docs/api-reference/text-openai-api) at `https://api.minimax.io/v1`, so we can use the `openai` Python SDK to call it.

## Requirements

1. Serverless Weaviate cluster (for the Weaviate Embedding Service)
    - You can create a 14-day free sandbox on [WCD](https://console.weaviate.cloud/)
2. MiniMax API key — sign up at [platform.minimax.io](https://platform.minimax.io/)

In [ ]:
!pip install -q weaviate-client openai

In [ ]:
import os
import re
import json

import weaviate
import weaviate.classes.config as wc
from weaviate.classes.config import Property, DataType
from weaviate.util import get_valid_uuid
from uuid import uuid4
from openai import OpenAI

## Configure Credentials

In [ ]:
WCD_URL = os.environ["WEAVIATE_URL"]       # Your Weaviate Cloud cluster URL
WCD_AUTH_KEY = os.environ["WEAVIATE_AUTH"] # Your Weaviate Cloud API key
MINIMAX_API_KEY = os.environ["MINIMAX_API_KEY"]  # Your MiniMax API key

## Connect to Weaviate

In [ ]:
client = weaviate.connect_to_wcs(
    cluster_url=WCD_URL,
    auth_credentials=weaviate.auth.AuthApiKey(WCD_AUTH_KEY),
)

print(client.is_ready())

## Set Up MiniMax Client

MiniMax provides an OpenAI-compatible API. We configure the `openai` SDK to point to MiniMax's base URL.

Available MiniMax chat models:
| Model | Description |
|-------|-------------|
| `MiniMax-M2.7` | Peak Performance. Ultimate Value. Master the Complex. |
| `MiniMax-M2.7-highspeed` | Same performance, faster and more agile. |

API reference: https://platform.minimax.io/docs/api-reference/text-openai-api

In [ ]:
minimax_client = OpenAI(
    api_key=MINIMAX_API_KEY,
    base_url="https://api.minimax.io/v1",
)

MINIMAX_MODEL = "MiniMax-M2.7"

## Create a Weaviate Collection

We use Weaviate's built-in embedding service (`text2vec_weaviate`) so we only need the MiniMax key for generation — no separate embedding API key required.

In [ ]:
# Note: in practice, you shouldn't rerun this cell, as it deletes your data
# in "BlogChunks", and then you need to re-import it again.

if client.collections.exists("BlogChunks"):
    client.collections.delete("BlogChunks")

client.collections.create(
    name="BlogChunks",
    vectorizer_config=wc.Configure.Vectorizer.text2vec_weaviate(
        model="Snowflake/snowflake-arctic-embed-l-v2.0",
    ),
    properties=[
        Property(name="content", data_type=DataType.TEXT)
    ]
)

print("Successfully created collection: BlogChunks.")

## Chunk and Import Data

We read Weaviate blog posts from the shared `./data` folder, split them into sentence chunks, and import them into the collection.

In [ ]:
def chunk_list(lst, chunk_size):
    """Break a list into chunks of the specified size."""
    return [lst[i:i + chunk_size] for i in range(0, len(lst), chunk_size)]

def split_into_sentences(text):
    """Split text into sentences using regular expressions."""
    sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?)\s', text)
    return [sentence.strip() for sentence in sentences if sentence.strip()]

def read_and_chunk_index_files(main_folder_path):
    """Read .md/.mdx files, split into sentences, and chunk every 5 sentences."""
    blog_chunks = []
    for file_path in os.listdir(main_folder_path):
        index_file_path = os.path.join(main_folder_path, file_path)
        if not os.path.isfile(index_file_path):
            continue
        with open(index_file_path, 'r', encoding='utf-8') as file:
            content = file.read()
            sentences = split_into_sentences(content)
            sentence_chunks = chunk_list(sentences, 5)
            blog_chunks.extend([' '.join(chunk) for chunk in sentence_chunks])
    return blog_chunks

# The ./data folder is shared with other model-provider notebooks
main_folder_path = '../data'
blog_chunks = read_and_chunk_index_files(main_folder_path)
print(f"Total chunks: {len(blog_chunks)}")

In [ ]:
# Preview the first chunk
blog_chunks[0]

In [ ]:
blogs = client.collections.get("BlogChunks")

for blog_chunk in blog_chunks:
    blogs.data.insert(
        properties={"content": blog_chunk},
        uuid=get_valid_uuid(uuid4())
    )

print(f"Inserted {len(blog_chunks)} chunks.")

## Retrieval — Hybrid Search

Hybrid search combines BM25 keyword search with vector similarity search.

- `alpha = 0` → pure BM25
- `alpha = 0.5` → half BM25, half vector search
- `alpha = 1` → pure vector search

In [ ]:
query = "What is Ref2Vec?"

blogs = client.collections.get("BlogChunks")
response = blogs.query.hybrid(query=query, alpha=0.5, limit=3)

retrieved_chunks = [obj.properties["content"] for obj in response.objects]
for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"--- Chunk {i} ---")
    print(chunk[:300])
    print()

## Generation — MiniMax RAG

We pass the retrieved chunks to MiniMax `MiniMax-M2.7` to generate a concise answer.

> **Note:** MiniMax requires `temperature` in the range `(0.0, 1.0]`. The default is `1.0`.

In [ ]:
def rag_with_minimax(query: str, chunks: list[str], model: str = MINIMAX_MODEL) -> str:
    """Retrieve context from Weaviate and generate an answer using MiniMax."""
    context = "\n\n".join(chunks)
    prompt = (
        f"Answer the following question based only on the provided context.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        f"Answer:"
    )

    completion = minimax_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Answer concisely based on the provided context."},
            {"role": "user", "content": prompt},
        ],
        temperature=1.0,  # MiniMax requires temperature in (0.0, 1.0], default 1.0
        max_tokens=512,
    )
    return completion.choices[0].message.content


answer = rag_with_minimax(query, retrieved_chunks)
print(f"Question: {query}\n")
print(f"MiniMax Answer:\n{answer}")

## Full RAG Pipeline

The helper below combines retrieval and generation in one call.

In [ ]:
def weaviate_minimax_rag(query: str, top_k: int = 3) -> str:
    """End-to-end RAG: retrieve from Weaviate, generate with MiniMax."""
    blogs = client.collections.get("BlogChunks")
    response = blogs.query.hybrid(query=query, alpha=0.5, limit=top_k)
    chunks = [obj.properties["content"] for obj in response.objects]
    return rag_with_minimax(query, chunks)


questions = [
    "How does hybrid search work in Weaviate?",
    "What are the use cases for Ref2Vec?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {weaviate_minimax_rag(q)}")
    print()

In [ ]:
client.close()